In [4]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# 1. Project root = one level up from notebooks/
PROJECT_ROOT = Path.cwd().parent

# 2. Define all paths
DATA_RAW       = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS        = PROJECT_ROOT / "reports"
FIGURES        = PROJECT_ROOT / "reports" / "figures"
MODELS         = PROJECT_ROOT / "models"

for p in [DATA_RAW, DATA_PROCESSED, REPORTS, FIGURES, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /Users/waiwai/Fake-News-Detector


In [6]:
# README file path
README_PATH = PROJECT_ROOT / "README.md"
readme_text = """# Fake-News-Detector

NLP pipeline that classifies news articles as fake or real
using spaCy text cleaning, TF-IDF features and Logistic Regression.

## Dataset
ISOT Fake News Dataset (Ahmed, Traore & Saad, 2017), from Kaggle:
"Fake and Real News Dataset" by Clément Bisaillon.
- Fake.csv: fake news articles (label 0)
- True.csv: real news articles from Reuters (label 1)

The data is not stored in this repo. Place both files in data/raw/.
"""

if README_PATH.exists() and README_PATH.read_text().strip() != "# Fake-News-Detector":
    print("README already has content. Not overwritten.")
else:
    README_PATH.write_text(readme_text)
    print("README written:", README_PATH)

README written: /Users/waiwai/Fake-News-Detector/README.md


In [8]:
# Requirements file path
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"
packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "spacy",
    "matplotlib",
    "seaborn",
    "jupyter",
    "joblib",
    "kagglehub",
]
REQUIREMENTS_PATH.write_text("\n".join(packages) + "\n")
print("Requirements written:", REQUIREMENTS_PATH)

Requirements written: /Users/waiwai/Fake-News-Detector/requirements.txt


In [9]:
import numpy as np
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print(f"Reproducibility configured. Global Seed = {SEED}")

Reproducibility configured. Global Seed = 42


In [11]:
FAKE_FILE = "Fake.csv"
TRUE_FILE = "True.csv"

COMBINED_FILE = "news_combined.csv"

FAKE_LABEL = 0
TRUE_LABEL = 1

SEED = 42

# Loading the Dataset

In [12]:
fake = pd.read_csv(DATA_RAW / FAKE_FILE)
true = pd.read_csv(DATA_RAW / TRUE_FILE)

size_table = pd.DataFrame({
    "file": [FAKE_FILE, TRUE_FILE],
    "rows": [fake.shape[0], true.shape[0]],
    "columns": [fake.shape[1], true.shape[1]],
})
size_table

,file,rows,columns
0,Fake.csv,23481,4
1,True.csv,21417,4


In [13]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [14]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [15]:
# Add a label column to each table
fake["label"] = FAKE_LABEL
true["label"] = TRUE_LABEL

# Stack the two tables on top of each other
df = pd.concat([fake, true], ignore_index=True)

# Count how many articles are in each class
df["label"].value_counts().to_frame()

,count
label,
0,23481
1,21417


In [16]:
df.dtypes.to_frame(name="dtype")

,dtype
title,str
text,str
subject,str
date,str
label,int64


In [17]:
missing = df.isna().sum().to_frame(name="missing")
missing["missing_%"] = (missing["missing"] / len(df) * 100).round(2)

missing

,missing,missing_%
title,0,0.0
text,0,0.0
subject,0,0.0
date,0,0.0
label,0,0.0


In [18]:
# Text columns to check
text_columns = ["title", "text", "subject", "date"]

# Count cells that are empty or only spaces
blank_counts = {}
for col in text_columns:
    blank_counts[col] = (df[col].str.strip() == "").sum()

pd.DataFrame.from_dict(blank_counts, orient="index", columns=["blank"])

,blank
title,0
text,631
subject,0
date,0


In [19]:
is_blank = df["text"].str.strip() == ""

df[is_blank]["label"].value_counts().to_frame(name="blank_texts")

,blank_texts
label,
0,630
1,1


In [20]:
df.duplicated().sum()

np.int64(209)

In [21]:
df.duplicated(subset=["text"]).sum()

np.int64(6252)

In [22]:
# For each text, count how many different labels it has
labels_per_text = df.groupby("text")["label"].nunique()

# How many texts have both labels (0 and 1)?
(labels_per_text > 1).sum()

np.int64(1)

In [23]:

df.to_csv(DATA_PROCESSED / COMBINED_FILE, index=False)

print("Saved:", DATA_PROCESSED / COMBINED_FILE)

Saved: /Users/waiwai/Fake-News-Detector/data/processed/news_combined.csv
